<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/module340/Lab6.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Lab 6 — Complete VQE for the H₂ Ground-State Energy
**Quantum Optimization and Simulation — VQE Laboratory Series**

The centerpiece lab: state preparation, Pauli measurements, optimization, and exact comparison.

**Suggested use:** 10–15 minute instructor demonstration followed by approximately one hour of independent work.

**Notebook style:** Most code is supplied. Complete the small items marked **YOUR TURN** and answer the reflection questions.

> Qiskit displays measured bitstrings as `q_(n-1)...q_0`. When orbital labels are written in the order `q0, q1, ...`, this notebook explicitly notes the convention.

## Learning objectives
- Build a complete reduced two-qubit H₂ VQE.
- Estimate each Pauli contribution using finite shots.
- Minimize the total energy with COBYLA.
- Compare VQE with exact diagonalization.

In [ ]:
# Run once in a fresh Google Colab session.
%pip -q install "qiskit~=2.5" "qiskit-aer~=0.17" "qiskit-algorithms~=0.4" "qiskit-nature~=0.8"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit_aer import AerSimulator

SEED = 123
SHOTS = 4096

def run_counts(qc, shots=SHOTS, noise_model=None):
    backend = AerSimulator(noise_model=noise_model)
    tqc = transpile(qc, backend, optimization_level=1)
    result = backend.run(tqc, shots=shots, seed_simulator=SEED).result()
    return result.get_counts()

def q0_first(qiskit_bits):
    return qiskit_bits.replace(" ", "")[::-1]

In [ ]:
from scipy.optimize import minimize

# Supplied reduced H2 Hamiltonian coefficients.
c0 = -1.0523732458
c1 = +0.3979374248
c2 = -0.3979374248
c3 = -0.0112801043
c4 = +0.1809311998

def h2_ansatz(theta):
    qc = QuantumCircuit(2)
    qc.x(0)
    qc.rxx(theta, 0, 1)
    qc.ryy(theta, 0, 1)
    return qc

In [ ]:
def parity_expectation(counts, qubits):
    total = sum(counts.values())
    expval = 0.0

    for bits, count in counts.items():
        bits = bits.replace(" ", "")
        value = 1
        for q in qubits:
            bit = int(bits[-1-q])
            value *= (1 if bit == 0 else -1)
        expval += value * count / total

    return expval

def measure_group(theta, basis):
    qc = h2_ansatz(theta)

    if basis == "X":
        qc.h([0,1])
    elif basis == "Y":
        qc.sdg([0,1])
        qc.h([0,1])

    qc.measure_all()
    return run_counts(qc, shots=4096)

## Part A — Energy function

In [ ]:
evaluation_history = []

def energy_from_shots(theta_array):
    theta = float(np.atleast_1d(theta_array)[0])

    z_counts = measure_group(theta, "Z")
    x_counts = measure_group(theta, "X")
    y_counts = measure_group(theta, "Y")

    z0 = parity_expectation(z_counts, [0])
    z1 = parity_expectation(z_counts, [1])
    zz = parity_expectation(z_counts, [0,1])
    xx = parity_expectation(x_counts, [0,1])
    yy = parity_expectation(y_counts, [0,1])

    energy = c0 + c1*z0 + c2*z1 + c3*zz + c4*(xx + yy)
    evaluation_history.append((theta, energy))
    return energy

## Part B — Scan the energy landscape

In [ ]:
scan_thetas = np.linspace(-np.pi/2, np.pi/2, 31)
scan_energies = [energy_from_shots([t]) for t in scan_thetas]

plt.plot(scan_thetas, scan_energies, "o-")
plt.xlabel("theta")
plt.ylabel("Energy (Hartree)")
plt.title("H2 VQE energy landscape")
plt.show()

## Part C — Optimize with COBYLA

In [ ]:
evaluation_history.clear()

result = minimize(
    energy_from_shots,
    x0=np.array([0.0]),
    method="COBYLA",
    options={"maxiter": 45, "rhobeg": 0.4}
)

print(result)
print("Optimal theta:", result.x[0])
print("VQE energy:", result.fun, "Hartree")

hist = np.array(evaluation_history)
plt.plot(hist[:,1], marker=".")
plt.xlabel("Energy evaluation")
plt.ylabel("Energy (Hartree)")
plt.title("COBYLA VQE convergence")
plt.show()

## Part D — Exact reference

In [ ]:
H = SparsePauliOp.from_list([
    ("II", c0),
    ("IZ", c1),
    ("ZI", c2),
    ("ZZ", c3),
    ("XX", c4),
    ("YY", c4),
])

exact_evals = np.linalg.eigvalsh(H.to_matrix())
exact_ground = exact_evals[0]

print("Exact eigenvalues:", exact_evals)
print("Exact ground-state energy:", exact_ground)
print("VQE error:", result.fun - exact_ground, "Hartree")

## Part E — Report the optimized mix

In [ ]:
theta_opt = result.x[0]
qc_opt = h2_ansatz(theta_opt)
qc_opt.measure_all()
counts = run_counts(qc_opt, shots=8192)
print(counts)
plot_histogram(counts)

### YOUR TURN
Convert the optimized energy from Hartree to electron-volts using \(1	ext{ Ha}=27.2114	ext{ eV}\).

## Reflection
Why can a shot-based VQE energy occasionally appear slightly below the exact ground-state energy even though the variational principle says it should not?

<details>
<summary><b>Instructor solution / suggested answer</b></summary>


    ```python
    energy_ev = result.fun * 27.2114
    ```

    The exact expectation value of a normalized trial state cannot be below the true ground-state energy. A finite-shot estimate contains statistical error, so the *estimate* can fluctuate slightly below the exact value even though the underlying state does not violate the variational principle.

</details>

## Optional — Generate H₂ with PySCF and Qiskit Nature

In [ ]:
# This optional section requires PySCF, which may take longer to install.
# %pip -q install pyscf
#
# from qiskit_nature.second_q.drivers import PySCFDriver
# from qiskit_nature.second_q.mappers import JordanWignerMapper
#
# driver = PySCFDriver(
#     atom="H 0 0 0; H 0 0 0.735",
#     basis="sto3g"
# )
# problem = driver.run()
# second_q_hamiltonian = problem.hamiltonian.second_q_op()
# qubit_hamiltonian_4q = JordanWignerMapper().map(second_q_hamiltonian)
#
# print(problem.num_spatial_orbitals, problem.num_particles)
# print(qubit_hamiltonian_4q)